### Import Dependecies

In [2]:
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

from langchain_core.messages import AIMessage, ToolMessage, SystemMessage
from langchain_core.messages import convert_to_openai_messages, convert_to_messages

from jinja2 import Template
from typing import Literal, Dict, Any, Annotated, List
from IPython.display import Image, display
from operator import add
from openai import OpenAI
import openai

import random
import ast
import inspect
import instructor
import json

from utils.utils import get_tool_descriptions, format_ai_message

from langsmith import traceable

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

import psycopg2
from psycopg2.extras import RealDictCursor
import numpy as np

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import Filter, FieldCondition, MatchValue, VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery

### Add to Shopping Cart Tool

In [11]:
items = [
    {
        "product_id": "B0B96LV4C5",
        "quantity": 2,
    },
    {
        "product_id": "B0BYYLJRHT",
        "quantity": 4,
    }
]

In [12]:
def add_to_shopping_cart(items: list[dict], user_id: str, cart_id: str) -> str: 

    """Add a list of provided items to the shopping cart. 

    Args: 
        items: A list of items to add to the shopping cart. Each item is a dictionary with the following keys: product_id, quantity. 
        user_id: The id of the user to add the items to the shopping cart.
        cart_id: The id of the shopping cart to add the items to.

    Returns:
        A list of the items added to the shopping cart.
    """

    conn = psycopg2.connect(
        host="localhost", 
        port=5433, 
        database="tools_database", 
        user="langgraph_user", 
        password="langgraph_password"
    )
    conn.autocommit = True

    with conn.cursor(cursor_factory=RealDictCursor) as cursor:

        for item in items:
            product_id = item['product_id']
            quantity = item['quantity']

            qdrant_client = QdrantClient(url="http://localhost:6333")

            dummy_vector = np.zeros(1536).tolist()
            payload = qdrant_client.query_points(
                collection_name="Amazon-items-collection-01-hybrid-search",
                prefetch=[
                    Prefetch(
                        query=dummy_vector,
                        filter=Filter(
                            must=[
                                FieldCondition(
                                    key="parent_asin",
                                    match=MatchValue(value=product_id) 
                                )
                            ]
                        ),              
                        using="text-embedding-3-small",
                        limit=20
                    )
                ],
                query=FusionQuery(fusion="rrf"),
                limit=1,
            ).points[0].payload

            product_image_url = payload.get("image")
            price = payload.get("price")
            currency = "USD"

            # Check if item already exist
            check_query = """
                SELECT id, quantity, price
                FROM shopping_carts.shopping_cart_items
                WHERE user_id = %s AND shopping_cart_id = %s AND product_id = %s
            """
            cursor.execute(check_query, (user_id, cart_id, product_id))
            existing_item = cursor.fetchone()

            if existing_item:
                # Update existing item
                new_quantity = existing_item['quantity'] + quantity

                update_query = """ 
                    UPDATE shopping_carts.shopping_cart_items
                    SET
                        quantity = %s,
                        price = %s,
                        currency = %s,
                        product_image_url = COALESCE(%s, product_image_url)
                    WHERE user_id = %s AND shopping_cart_id = %s AND product_id = %s
                    RETURNING id, quantity, price
                """

                cursor.execute(update_query, (new_quantity, price, currency, product_image_url, user_id, cart_id, product_id))

            else:
                # Insert new item
                insert_query = """ 
                    INSERT INTO shopping_carts.shopping_cart_items (
                        user_id, shopping_cart_id, product_id,
                        price, quantity, currency, product_image_url
                    ) VALUES (%s,%s,%s,%s,%s,%s,%s)
                    RETURNING id, quantity, price
                """

                cursor.execute(insert_query, (user_id, cart_id, product_id, price, quantity, currency, product_image_url))

    return f"Added {item} to the shopping cart."



In [13]:
add_to_shopping_cart(items, "user", "test")

"Added {'product_id': 'B0BYYLJRHT', 'quantity': 4} to the shopping cart."

In [14]:
items_2 = [
    {
        "product_id": "B0B96LV4C5",
        "quantity": 8,
    }
]

In [15]:
add_to_shopping_cart(items_2, "user", "test")

"Added {'product_id': 'B0B96LV4C5', 'quantity': 8} to the shopping cart."

### Get the Shopping Cart Items Tool

In [16]:
def get_shopping_cart(user_id: str, cart_id: str) -> list[dict]:
    """ 
    Retrieve all items in a user's shopping cart.

    Args:
        user_id: User ID
        cart_id: Cart ID

    Returns:
        List of dictionaries containing cart items
    """

    conn = psycopg2.connect(
        host="localhost", 
        port=5433, 
        database="tools_database", 
        user="langgraph_user", 
        password="langgraph_password"
    )
    conn.autocommit = True

    with conn.cursor(cursor_factory=RealDictCursor) as cursor:

        query = """
            SELECT 
                product_id, price, quantity,
                currency, product_image_url,
                (quantity * price) as total_price
            FROM shopping_carts.shopping_cart_items
            WHERE user_id = %s AND shopping_cart_id = %s
            ORDER BY added_at DESC
        """
        cursor.execute(query, (user_id, cart_id))

        return [dict(row) for row in cursor.fetchall()]

In [17]:
get_shopping_cart("user", "test")

[{'product_id': 'B0BYYLJRHT',
  'price': Decimal('12.59'),
  'quantity': 4,
  'currency': 'USD',
  'product_image_url': 'https://m.media-amazon.com/images/I/41lk7yDhf3L.jpg',
  'total_price': Decimal('50.36')},
 {'product_id': 'B0B96LV4C5',
  'price': Decimal('14.99'),
  'quantity': 10,
  'currency': 'USD',
  'product_image_url': 'https://m.media-amazon.com/images/I/41J3p7fyJHL.jpg',
  'total_price': Decimal('149.90')}]

### Deleting Items from the Shopping Cart Tool

In [18]:
def remove_from_cart(product_id: str, user_id: str, cart_id: str) -> str:
    """ 
    Remove an item completely from the shopping cart.

    Args:
        user_id: User ID
        product_id: Product ID to remove
        cart_id: Cart ID

    Returns:
        True if item was removed, False if item wasn't found.
    """

    conn = psycopg2.connect(
        host="localhost", 
        port=5433, 
        database="tools_database", 
        user="langgraph_user", 
        password="langgraph_password"
    )
    conn.autocommit = True

    with conn.cursor(cursor_factory=RealDictCursor) as cursor:

        query = """
            DELETE FROM shopping_carts.shopping_cart_items
            WHERE user_id = %s AND shopping_cart_id = %s AND product_id = %s
        """
        cursor.execute(query, (user_id, cart_id, product_id))

        return cursor.rowcount > 0

In [19]:
remove_from_cart("B0BYYLJRHT", "user", "test")

True

In [20]:
remove_from_cart("B0BYYLJRHT", "user", "test")

False